<a href="https://colab.research.google.com/github/Nayab-Gauhar/Using-GAN-to-generate-the-FM-matrix-Autism-/blob/main/autism_gan_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autism GAN Pipeline — Complete 3-Step Notebook

**Step 1:** Download ABIDE-I data & build FC matrices
**Step 2:** Train conditional WGAN-GP to synthesize FC matrices
**Step 3:** Classify ASD vs Control using real + synthetic data

## Step 0 — Install Dependencies

In [6]:
import subprocess, sys
for pkg in ["nilearn", "numpy", "torch", "scikit-learn", "matplotlib", "seaborn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

## Step 1 — Build FC Matrix Dataset from ABIDE-I

Downloads preprocessed ABIDE-I ROI time series via `nilearn` and computes
Pearson correlation functional connectivity (FC) matrices.

In [7]:
import os
import numpy as np
from nilearn import datasets
from nilearn.connectome import ConnectivityMeasure

# ---- config ----
ATLAS = "cc200"          # 'aal', 'ho', 'ez', 'tt', 'dosenbach160', 'cc200'
N_SUBJECTS = None        # None = all available subjects
OUT_DIR = "./fc_data"
DATA_DIR = "./abide_raw"

os.makedirs(OUT_DIR, exist_ok=True)

### Fetch ROI time series (CPAC pipeline, no bandpass, no GSR — matches paper)

In [8]:
from pathlib import Path
from nilearn import datasets

roi_dir = Path(DATA_DIR) / "ABIDE_pcp" / "cpac" / "nofilt_noglobal" / f"rois_{ATLAS}"

if roi_dir.exists() and any(roi_dir.glob("*.1D")):
    print("✅ ABIDE dataset already downloaded.")
    print("Loading dataset metadata...\n")
else:
    print(f"Fetching ABIDE-I ROI time series for atlas={ATLAS} ...")
    print("(First run downloads ~800+ files from AWS — may take 30-60 min)\n")

abide = datasets.fetch_abide_pcp(
    data_dir=DATA_DIR,
    pipeline="cpac",
    band_pass_filtering=False,
    global_signal_regression=False,
    derivatives=[f"rois_{ATLAS}"],
    n_subjects=N_SUBJECTS,
    quality_checked=True,
)

roi_key = f"rois_{ATLAS}"
time_series_list = abide[roi_key]
phenotypic = abide.phenotypic

print(f"\nFetched {len(time_series_list)} subjects")
print(f"Phenotypic columns: {list(phenotypic.columns[:10])}...")

Fetching ABIDE-I ROI time series for atlas=cc200 ...
(First run downloads ~800+ files from AWS — may take 30-60 min)



[fetch_abide_pcp] Added README.md to ./abide_raw

[fetch_abide_pcp] Dataset created in abide_raw/ABIDE_pcp

[fetch_abide_pcp] Downloading data from 
https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Phenotypic_V1_0b_preprocessed1.csv ...

[fetch_abide_pcp]  ...done. (2 seconds, 0 min)

...

[fetch_abide_pcp] Downloading data from 
https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Outputs/cpac/nofilt_noglobal/rois_cc200/MaxMun_a_0
051606_rois_cc200.1D ...

[fetch_abide_pcp]  ...done. (1 seconds, 0 min)

[fetch_abide_pcp] Downloading data from 
https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Outputs/cpac/nofilt_noglobal/rois_cc200/MaxMun_a_0
051607_rois_cc200.1D ...

[fetch_abide_pcp]  ...done. (1 seconds, 0 min)


Fetched 871 subjects
Phenotypic columns: ['i', 'Unnamed: 0', 'SUB_ID', 'X', 'subject', 'SITE_ID', 'FILE_ID', 'DX_GROUP', 'DSM_IV_TR', 'AGE_AT_SCAN']...


### Compute FC matrices (Pearson correlation)

In [9]:
conn_measure = ConnectivityMeasure(kind="correlation")

fc_matrices, labels, sub_ids = [], [], []

for i, ts in enumerate(time_series_list):
    if ts is None or len(ts) == 0:
        continue
    fc = conn_measure.fit_transform([ts])[0]
    fc_matrices.append(fc.astype(np.float32))
    # BUG FIX: use .iloc[i] instead of [i] — phenotypic index starts at 1
    labels.append(phenotypic["DX_GROUP"].iloc[i])
    sub_ids.append(phenotypic["SUB_ID"].iloc[i])

fc_matrices = np.array(fc_matrices, dtype=np.float32)
labels = np.array(labels)
sub_ids = np.array(sub_ids)

print(f"FC matrices shape: {fc_matrices.shape}")

/tmp/ipykernel_10792/3603091549.py:8: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  fc = conn_measure.fit_transform([ts])[0]
/tmp/ipykernel_10792/3603091549.py:8: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, ple

FC matrices shape: (871, 200, 200)


### Save FC data

In [10]:
np.save(f"{OUT_DIR}/fc_matrices_{ATLAS}.npy", fc_matrices)
np.save(f"{OUT_DIR}/labels_{ATLAS}.npy", labels)
np.save(f"{OUT_DIR}/sub_ids_{ATLAS}.npy", sub_ids)

n_asd = int((labels == 1).sum())
n_ctrl = int((labels == 2).sum())
print(f"Saved {fc_matrices.shape[0]} FC matrices of shape {fc_matrices.shape[1:]}")
print(f"ASD: {n_asd}   Control: {n_ctrl}")

Saved 871 FC matrices of shape (200, 200)
ASD: 403   Control: 468


---
## Step 2 — Conditional WGAN-GP for Synthetic FC Matrices

Trains on the real FC matrices and generates class-conditional synthetic
ASD / Control FC matrices. Only upper-triangular correlations are modeled;
full symmetric matrices (diagonal = 1.0) are reconstructed after generation.

In [11]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---- config ----
LATENT_DIM = 128
BATCH_SIZE = 32
EPOCHS = 2000
N_CRITIC = 5
LAMBDA_GP = 10
LR = 1e-4
N_SYNTH_PER_CLASS = 200
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
np.random.seed(42)
print("Device:", DEVICE)

Device: cuda


### Load real FC matrices and flatten to upper-triangular vectors

In [12]:
fc = np.load(f"{OUT_DIR}/fc_matrices_{ATLAS}.npy")
raw_labels = np.load(f"{OUT_DIR}/labels_{ATLAS}.npy")
gan_labels = (raw_labels == 1).astype(np.float32)  # ASD -> 1.0, Control -> 0.0

N, R, _ = fc.shape
iu = np.triu_indices(R, k=1)
vec_dim = len(iu[0])
print(f"{N} subjects | ROI={R} | vector dim={vec_dim}")


def mat_to_vec(mats):
    return np.stack([m[iu] for m in mats])


def vec_to_mat(vecs, roi=R):
    """Reconstruct full symmetric FC matrix from upper-triangular vector."""
    mats = np.zeros((len(vecs), roi, roi), dtype=np.float32)
    for i, v in enumerate(vecs):
        m = np.zeros((roi, roi), dtype=np.float32)
        m[iu] = v
        m = m + m.T
        np.fill_diagonal(m, 1.0)
        mats[i] = m
    return mats


X = mat_to_vec(fc).astype(np.float32)

871 subjects | ROI=200 | vector dim=19900


### Dataset, Generator, Discriminator

In [13]:
class FCDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


dataset = FCDataset(X, gan_labels)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)


class Generator(nn.Module):
    def __init__(self, latent_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, out_dim), nn.Tanh(),
        )

    def forward(self, z, y):
        return self.net(torch.cat([z, y], dim=1))


class Discriminator(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim + 1, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x, y):
        return self.net(torch.cat([x, y], dim=1))


G = Generator(LATENT_DIM, vec_dim).to(DEVICE)
D = Discriminator(vec_dim).to(DEVICE)
g_opt = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.9))
d_opt = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.9))

### Gradient Penalty + Training Loop

In [ ]:
def gradient_penalty(D, real, fake, y):
    eps = torch.rand(real.size(0), 1, device=DEVICE)
    interp = (eps * real + (1 - eps) * fake).requires_grad_(True)
    d_interp = D(interp, y)
    grads = torch.autograd.grad(
        outputs=d_interp, inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True, retain_graph=True,
    )[0]
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


print(f"Training WGAN-GP for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    for real_x, real_y in loader:
        real_x, real_y = real_x.to(DEVICE), real_y.to(DEVICE)

        for _ in range(N_CRITIC):
            z = torch.randn(real_x.size(0), LATENT_DIM, device=DEVICE)
            fake_x = G(z, real_y).detach()
            d_real = D(real_x, real_y)
            d_fake = D(fake_x, real_y)
            gp = gradient_penalty(D, real_x, fake_x, real_y)
            d_loss = d_fake.mean() - d_real.mean() + LAMBDA_GP * gp
            d_opt.zero_grad(); d_loss.backward(); d_opt.step()

        z = torch.randn(real_x.size(0), LATENT_DIM, device=DEVICE)
        fake_x = G(z, real_y)
        g_loss = -D(fake_x, real_y).mean()
        g_opt.zero_grad(); g_loss.backward(); g_opt.step()

    if epoch % 100 == 0:
        print(f"  epoch {epoch:4d}/{EPOCHS} | d_loss={d_loss.item():.3f} | g_loss={g_loss.item():.3f}")

torch.save(G.state_dict(), f"generator_{ATLAS}.pt")
print("Generator saved.")

Training WGAN-GP for 2000 epochs...


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  epoch    0/2000 | d_loss=-18.685 | g_loss=-10.115
  epoch  100/2000 | d_loss=-4.897 | g_loss=10.611
  epoch  200/2000 | d_loss=-5.238 | g_loss=5.869


### Generate synthetic FC matrices

In [ ]:
def synthesize(n_per_class):
    G.eval()
    with torch.no_grad():
        z1 = torch.randn(n_per_class, LATENT_DIM, device=DEVICE)
        y1 = torch.ones(n_per_class, 1, device=DEVICE)
        z0 = torch.randn(n_per_class, LATENT_DIM, device=DEVICE)
        y0 = torch.zeros(n_per_class, 1, device=DEVICE)
        vec_asd = G(z1, y1).cpu().numpy()
        vec_ctrl = G(z0, y0).cpu().numpy()
    return vec_to_mat(vec_asd), vec_to_mat(vec_ctrl)


synth_asd, synth_ctrl = synthesize(N_SYNTH_PER_CLASS)
np.save(f"{OUT_DIR}/synth_asd_{ATLAS}.npy", synth_asd)
np.save(f"{OUT_DIR}/synth_ctrl_{ATLAS}.npy", synth_ctrl)

print(f"Generated {N_SYNTH_PER_CLASS} synthetic ASD + {N_SYNTH_PER_CLASS} control FC matrices")
print(f"Symmetry check (should be 0.0): {np.abs(synth_asd[0] - synth_asd[0].T).max():.6f}")
print(f"Diagonal check (should be True): {np.allclose(np.diagonal(synth_asd[0]), 1.0)}")

### Visualize real vs synthetic FC matrices

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titles = ["Real ASD", "Synthetic ASD", "Real Control", "Synthetic Control"]
matrices = [
    fc[raw_labels == 1][0],
    synth_asd[0],
    fc[raw_labels == 2][0],
    synth_ctrl[0],
]
for ax, mat, title in zip(axes, matrices, titles):
    im = ax.imshow(mat, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("ROI")
    ax.set_ylabel("ROI")
plt.colorbar(im, ax=axes, shrink=0.8, label="Correlation")
plt.suptitle("Functional Connectivity Matrices", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fc_comparison_{ATLAS}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved comparison plot to {OUT_DIR}/fc_comparison_{ATLAS}.png")

---
## Step 3 — Classification (ASD vs Control)

Trains multiple classifiers on:
1. **Real data only** (baseline)
2. **Real + GAN-synthesized data** (augmented)

Compares performance to show the effect of GAN augmentation.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import seaborn as sns

### Prepare real and augmented datasets

In [ ]:
# Load data
fc_real = np.load(f"{OUT_DIR}/fc_matrices_{ATLAS}.npy")
labels_real = np.load(f"{OUT_DIR}/labels_{ATLAS}.npy")
synth_asd = np.load(f"{OUT_DIR}/synth_asd_{ATLAS}.npy")
synth_ctrl = np.load(f"{OUT_DIR}/synth_ctrl_{ATLAS}.npy")

# Flatten FC matrices to upper-triangular vectors for classifiers
iu = np.triu_indices(fc_real.shape[1], k=1)

X_real = np.stack([m[iu] for m in fc_real])
y_real = (labels_real == 1).astype(int)  # 1=ASD, 0=Control

X_synth_asd = np.stack([m[iu] for m in synth_asd])
X_synth_ctrl = np.stack([m[iu] for m in synth_ctrl])

X_augmented = np.vstack([X_real, X_synth_asd, X_synth_ctrl])
y_augmented = np.concatenate([
    y_real,
    np.ones(len(X_synth_asd), dtype=int),
    np.zeros(len(X_synth_ctrl), dtype=int),
])

print(f"Real dataset:      {X_real.shape[0]} samples ({y_real.sum()} ASD, {(1-y_real).sum()} Ctrl)")
print(f"Augmented dataset: {X_augmented.shape[0]} samples")
print(f"Feature dim:       {X_real.shape[1]}")

### Define classifiers

In [ ]:
classifiers = {
    "Linear SVM": SVC(kernel="linear", C=1.0, probability=True, random_state=42),
    "RBF SVM": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    "Logistic Regression (L2)": LogisticRegression(penalty="l2", C=1.0, max_iter=1000, random_state=42),
    "LDA": LinearDiscriminantAnalysis(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

### Evaluate: Real-only vs Augmented (Stratified 10-Fold CV)

**Important:** We always test on REAL data only. The synthetic data is only
used for training in the augmented experiment.

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = {"Classifier": [], "Training Data": [], "Accuracy": [], "F1": [], "AUC": []}

for clf_name, clf_template in classifiers.items():
    for data_label, X_train_pool, y_train_pool in [
        ("Real Only", X_real, y_real),
        ("Real + Synthetic", X_augmented, y_augmented),
    ]:
        fold_acc, fold_f1, fold_auc = [], [], []

        for train_idx, test_idx in skf.split(X_real, y_real):
            # Test set is always from REAL data
            X_test, y_test = X_real[train_idx], y_real[train_idx]  # not used for training
            X_test, y_test = X_real[test_idx], y_real[test_idx]

            if data_label == "Real Only":
                X_tr, y_tr = X_real[train_idx], y_real[train_idx]
            else:
                # Real training fold + ALL synthetic data
                X_tr = np.vstack([X_real[train_idx], X_synth_asd, X_synth_ctrl])
                y_tr = np.concatenate([
                    y_real[train_idx],
                    np.ones(len(X_synth_asd), dtype=int),
                    np.zeros(len(X_synth_ctrl), dtype=int),
                ])

            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_te_scaled = scaler.transform(X_test)

            from sklearn.base import clone
            clf = clone(clf_template)
            clf.fit(X_tr_scaled, y_tr)

            y_pred = clf.predict(X_te_scaled)
            y_prob = clf.predict_proba(X_te_scaled)[:, 1] if hasattr(clf, "predict_proba") else y_pred.astype(float)

            fold_acc.append(accuracy_score(y_test, y_pred))
            fold_f1.append(f1_score(y_test, y_pred))
            fold_auc.append(roc_auc_score(y_test, y_prob))

        results["Classifier"].append(clf_name)
        results["Training Data"].append(data_label)
        results["Accuracy"].append(np.mean(fold_acc))
        results["F1"].append(np.mean(fold_f1))
        results["AUC"].append(np.mean(fold_auc))

        print(f"{clf_name:25s} | {data_label:17s} | Acc={np.mean(fold_acc):.4f} | F1={np.mean(fold_f1):.4f} | AUC={np.mean(fold_auc):.4f}")

### Results Table

In [ ]:
import pandas as pd

df_results = pd.DataFrame(results)
print("\n" + "="*80)
print("CLASSIFICATION RESULTS — 10-Fold Stratified Cross-Validation")
print("="*80)
print(df_results.to_string(index=False, float_format="%.4f"))

### Visualization — Real vs Augmented Performance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics = ["Accuracy", "F1", "AUC"]

for ax, metric in zip(axes, metrics):
    pivot = df_results.pivot(index="Classifier", columns="Training Data", values=metric)
    pivot.plot(kind="bar", ax=ax, rot=30, color=["#4A90D9", "#E74C3C"])
    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_ylabel(metric)
    ax.set_ylim(0.4, 1.0)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(f"ASD vs Control Classification — Real vs GAN-Augmented ({ATLAS})",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/classification_results_{ATLAS}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved results plot to {OUT_DIR}/classification_results_{ATLAS}.png")

### Best Classifier Summary

In [ ]:
print("\n" + "="*60)
best_real = df_results[df_results["Training Data"] == "Real Only"].sort_values("AUC", ascending=False).iloc[0]
best_aug = df_results[df_results["Training Data"] == "Real + Synthetic"].sort_values("AUC", ascending=False).iloc[0]

print(f"Best (Real Only):      {best_real['Classifier']} — AUC={best_real['AUC']:.4f}, Acc={best_real['Accuracy']:.4f}")
print(f"Best (Augmented):      {best_aug['Classifier']} — AUC={best_aug['AUC']:.4f}, Acc={best_aug['Accuracy']:.4f}")
print(f"AUC Improvement:       {best_aug['AUC'] - best_real['AUC']:+.4f}")
print("="*60)